# Data Lake Demo – Vollständig

### Offene Formate & ML-Workloads

| Format | Eigenschaft | ML-Relevanz |
|--------|-------------|-------------|
| **Parquet** | Spaltenorientiert, komprimiert | Direkt lesbar mit pandas, PyArrow, Spark, scikit-learn |
| **Delta Lake** | Parquet + Transaktionslog | ACID, Versionierung, Schema-Evolution |

Da offene Formate keinen proprietären Server benötigen, können ML-Frameworks Daten **direkt lesen** – kein ETL-Export nötig.

### Apache Spark vs. DuckDB

| | Apache Spark | DuckDB |
|---|---|---|
| Skala | Petabytes, verteiltes Cluster | Gigabytes, einzelner Prozess |
| Setup | JVM + Cluster-Manager | `pip install duckdb` |
| Delta Lake | Nativer Produktionsmotor (Databricks) | Extension + `deltalake`-Bibliothek |
| Typischer Einsatz | Produktions-ETL-Pipelines | Lokale Entwicklung, Analyse, Demos |

> **Hinweis:** Apache Spark ist der Standard-Produktionsmotor für Delta Lake. DuckDB bietet dieselben Konzepte ohne JVM oder Docker – ideal zum Lernen und für leichte Workloads.

### Setup

Wir laden DuckDB mit der Delta-Extension. `INSTALL delta; LOAD delta;` aktiviert das Lesen von Delta-Tabellen direkt in DuckDB. `deltalake` (delta-rs, in Rust geschrieben) übernimmt das Schreiben – kein Spark, keine JVM nötig.

In [22]:
import duckdb
import pandas as pd
import shutil, os
from deltalake import write_deltalake, DeltaTable

con = duckdb.connect()
con.sql("INSTALL delta; LOAD delta;")
print("DuckDB", duckdb.__version__, "bereit")

DuckDB 1.5.2 bereit


## 2. Rohdaten-Ingestion (CSV & JSON)

DuckDB liest CSV und JSON direkt per SQL – keine Vorverarbeitung nötig. Das entspricht dem Data-Lake-Prinzip: Rohdaten landen unverändert im See.

### CSV einlesen

DuckDB liest die CSV-Datei direkt per SQL ein – ohne Schema-Definition oder Import-Tool. Das ist Data-Lake-typisch: Rohdaten landen unverändert im See, Strukturierung passiert beim Lesen (*Schema-on-Read*). Die zweite Query bestätigt den Datenumfang: 1000 Flüge von Januar bis September 2025.

In [23]:
con.sql("CREATE OR REPLACE TABLE flights AS SELECT * FROM 'data/flights.csv'")
con.sql("SELECT COUNT(*) AS zeilen, MIN(datum) AS von, MAX(datum) AS bis FROM flights")

┌────────┬─────────────────────┬─────────────────────┐
│ zeilen │         von         │         bis         │
│ int64  │      timestamp      │      timestamp      │
├────────┼─────────────────────┼─────────────────────┤
│   1000 │ 2025-01-01 00:00:00 │ 2025-09-07 18:00:00 │
└────────┴─────────────────────┴─────────────────────┘

### JSON einlesen

JSON ist verschachtelt – die Flughäfen liegen als Array unter dem Schlüssel `airports`. `read_json_auto()` erkennt das Schema automatisch, `UNNEST` flacht das Array in einzelne Zeilen ab. So wird aus halbstrukturiertem JSON eine relationale Tabelle, ohne Vorverarbeitung in Python.

In [24]:
con.sql("""
    CREATE OR REPLACE TABLE airports AS
    SELECT airport.code, airport.name, airport.city, airport.country
    FROM read_json_auto('data/airports.json') t,
    UNNEST(t.airports) AS u(airport)
""")
con.sql("SELECT * FROM airports ORDER BY code")

┌─────────┬───────────────────────────────────────┬────────────┬─────────┐
│  code   │                 name                  │    city    │ country │
│ varchar │                varchar                │  varchar   │ varchar │
├─────────┼───────────────────────────────────────┼────────────┼─────────┤
│ BER     │ Berlin Brandenburg Airport            │ Berlin     │ DE      │
│ CDG     │ Charles de Gaulle Airport             │ Paris      │ FR      │
│ DUS     │ Düsseldorf Airport                    │ Düsseldorf │ DE      │
│ DXB     │ Dubai International Airport           │ Dubai      │ AE      │
│ FRA     │ Frankfurt Airport                     │ Frankfurt  │ DE      │
│ HAM     │ Hamburg Airport                       │ Hamburg    │ DE      │
│ HND     │ Haneda Airport                        │ Tokyo      │ JP      │
│ JFK     │ John F. Kennedy International Airport │ New York   │ US      │
│ LHR     │ Heathrow Airport                      │ London     │ UK      │
│ MUC     │ Munich Airpor

## 3. Konvertierung in offene Formate

**Parquet** ist das Standard-Speicherformat im Data Lake: spaltenorientiert, komprimiert und plattformunabhängig. Partitionierung nach Jahr/Monat beschleunigt zeitbasierte Abfragen durch **Partition Pruning** – nur relevante Dateien werden gelesen.

### Partitioniertes Parquet schreiben

`COPY ... TO ... (FORMAT PARQUET, PARTITION_BY (year, month))` schreibt die Flugdaten in eine Ordnerhierarchie wie `year=2025/month=3/`. Bei späteren Abfragen mit Zeitfilter überspringt DuckDB irrelevante Ordner komplett – das ist **Partition Pruning** und der Hauptgrund für Partitionierung.

In [25]:
# Ausgabeverzeichnis zurücksetzen
if os.path.exists('data/flights_partitioned_new'):
    shutil.rmtree('data/flights_partitioned_new')

con.sql("""
COPY (
    SELECT *, year(datum) AS year, month(datum) AS month
    FROM flights
)
TO 'data/flights_partitioned_new'
(FORMAT PARQUET, PARTITION_BY (year, month));
""")
print("Partitionierte Parquet-Dateien geschrieben")

Partitionierte Parquet-Dateien geschrieben


### Einzelne Parquet-Datei

Kleine Tabellen brauchen keine Partitionierung – eine einzige Parquet-Datei reicht. Die zweite Query liest direkt aus der Datei (`FROM 'data/airports_new.parquet'`), ohne dass eine Tabelle in DuckDB registriert sein muss. Genau das meint *offenes Format*: die Datei ist die Datenbank.

In [26]:
con.sql("COPY airports TO 'data/airports_new.parquet' (FORMAT PARQUET)")

# Direkt aus Parquet lesen – kein DuckDB-Server nötig (open format!)
con.sql("SELECT * FROM 'data/airports_new.parquet' ORDER BY code")

┌─────────┬───────────────────────────────────────┬────────────┬─────────┐
│  code   │                 name                  │    city    │ country │
│ varchar │                varchar                │  varchar   │ varchar │
├─────────┼───────────────────────────────────────┼────────────┼─────────┤
│ BER     │ Berlin Brandenburg Airport            │ Berlin     │ DE      │
│ CDG     │ Charles de Gaulle Airport             │ Paris      │ FR      │
│ DUS     │ Düsseldorf Airport                    │ Düsseldorf │ DE      │
│ DXB     │ Dubai International Airport           │ Dubai      │ AE      │
│ FRA     │ Frankfurt Airport                     │ Frankfurt  │ DE      │
│ HAM     │ Hamburg Airport                       │ Hamburg    │ DE      │
│ HND     │ Haneda Airport                        │ Tokyo      │ JP      │
│ JFK     │ John F. Kennedy International Airport │ New York   │ US      │
│ LHR     │ Heathrow Airport                      │ London     │ UK      │
│ MUC     │ Munich Airpor

## 4. Analytics SQL Queries

DuckDB führt SQL direkt auf Parquet-Dateien aus – keine Datenbank-Einrichtung nötig.

In [27]:
# Flüge pro Zielflughafen mit vollständigen Metadaten (JOIN)
con.sql("""
SELECT
    f.dest,
    a.name  AS airport_name,
    a.country,
    COUNT(*) AS flights
FROM flights f
JOIN airports a ON f.dest = a.code
GROUP BY f.dest, a.name, a.country
ORDER BY flights DESC
""")

┌─────────┬───────────────────────────────────────┬─────────┬─────────┐
│  dest   │             airport_name              │ country │ flights │
│ varchar │                varchar                │ varchar │  int64  │
├─────────┼───────────────────────────────────────┼─────────┼─────────┤
│ HND     │ Haneda Airport                        │ JP      │     176 │
│ SIN     │ Changi Airport                        │ SG      │     169 │
│ LHR     │ Heathrow Airport                      │ UK      │     166 │
│ DXB     │ Dubai International Airport           │ AE      │     166 │
│ JFK     │ John F. Kennedy International Airport │ US      │     162 │
│ CDG     │ Charles de Gaulle Airport             │ FR      │     161 │
└─────────┴───────────────────────────────────────┴─────────┴─────────┘

In [30]:
# Monatlicher Flug-Trend direkt auf partitioniertem Parquet
# Partition Pruning: DuckDB liest nur die benötigten Ordner
con.sql("""
SELECT year, month, COUNT(*) AS flights
FROM read_parquet('data/flights_partitioned_new/**/*.parquet', hive_partitioning = true)
GROUP BY year, month
ORDER BY year, month
""")

┌───────┬───────┬─────────┐
│ year  │ month │ flights │
│ int64 │ int64 │  int64  │
├───────┼───────┼─────────┤
│  2025 │     1 │     124 │
│  2025 │     2 │     112 │
│  2025 │     3 │     124 │
│  2025 │     4 │     120 │
│  2025 │     5 │     124 │
│  2025 │     6 │     120 │
│  2025 │     7 │     124 │
│  2025 │     8 │     124 │
│  2025 │     9 │      28 │
└───────┴───────┴─────────┘

## 5. Delta Lake & ACID-Transaktionen

**Delta Lake** erweitert Parquet um einen Transaktionslog (`_delta_log/`). Jede Schreiboperation erzeugt eine neue JSON-Datei im Log – das garantiert ACID:

| Eigenschaft | Bedeutung | Delta-Mechanismus |
|---|---|---|
| **A**tomicity | Alles oder nichts | Log-Eintrag nur nach vollständigem Schreiben |
| **C**onsistency | Daten bleiben gültig | Schema-Validierung beim Commit |
| **I**solation | Lesende sehen konsistenten Snapshot | Snapshot-Isolation via Versionsnummer |
| **D**urability | Einmal committed = dauerhaft | Log-Datei ist der dauerhafte Beweis |

Jede `write_deltalake()`-Operation erzeugt eine neue Version – Basis für **Time Travel**.

### Delta Version 0 anlegen

Erster Schreibvorgang: nur die Q1-Flüge. `write_deltalake()` legt zwei Dinge an – die Parquet-Datei mit den Daten *und* den Ordner `_delta_log/` mit dem Transaktionslog. Damit existiert Version 0 der Tabelle.

In [32]:
# Delta-Tabelle neu anlegen
if os.path.exists('data/flights_delta'):
    shutil.rmtree('data/flights_delta')

# Version 0: Flüge Q1 (Jan–Mär)
flights_q1 = con.sql("SELECT * FROM flights WHERE month(datum) BETWEEN 1 AND 3").df()
write_deltalake('data/flights_delta', flights_q1)
print(f"Version 0 geschrieben: {len(flights_q1)} Zeilen (Q1)")

Version 0 geschrieben: 360 Zeilen (Q1)


### Append → Version 1

Mit `mode='append'` werden die Q2-Flüge hinzugefügt. Die alten Daten bleiben unangetastet – Delta legt nur eine neue Parquet-Datei und einen neuen Log-Eintrag an. Ergebnis: Version 1 mit 724 Zeilen, Version 0 weiterhin abrufbar.

In [33]:
# Version 1: Q2-Flüge anhängen
flights_q2 = con.sql("SELECT * FROM flights WHERE month(datum) BETWEEN 4 AND 6").df()
write_deltalake('data/flights_delta', flights_q2, mode='append')
print(f"Version 1 geschrieben: +{len(flights_q2)} Zeilen (Q2)")

Version 1 geschrieben: +364 Zeilen (Q2)


### Append → Version 2

Dasselbe noch einmal mit Q3. Nach diesem Schritt existieren drei abrufbare Snapshots der Tabelle (v0, v1, v2). Jede Version ist atomar entstanden – entweder ist der Log-Eintrag da oder nicht, ein halber Zustand kann nicht auftreten.

In [34]:
# Version 2: Q3-Flüge anhängen
flights_q3 = con.sql("SELECT * FROM flights WHERE month(datum) BETWEEN 7 AND 9").df()
write_deltalake('data/flights_delta', flights_q3, mode='append')
print(f"Version 2 geschrieben: +{len(flights_q3)} Zeilen (Q3)")

Version 2 geschrieben: +276 Zeilen (Q3)


### Versionshistorie aus dem Log

`dt.history()` liest den Transaktionslog und listet alle Commits in umgekehrter Reihenfolge auf. Drei Einträge = drei `WRITE`-Operationen. Das ist der direkte ACID-Beweis: jeder dieser Einträge entspricht einer atomaren Transaktion, die Zeitstempel zeigen die Reihenfolge.

In [35]:
# ACID-Beweis: Den Transaktionslog direkt auslesen
# Jede Datei in _delta_log/ entspricht einem atomaren Commit
dt = DeltaTable('data/flights_delta')
print("=== Delta-Versionshistorie (Transaktionslog) ===")
for h in dt.history():
    print(f"  Version {h['version']}  |  {h['timestamp']}  |  Operation: {h.get('operation', '?')}")

=== Delta-Versionshistorie (Transaktionslog) ===
  Version 2  |  1777886706957  |  Operation: WRITE
  Version 1  |  1777886706925  |  Operation: WRITE
  Version 0  |  1777886706892  |  Operation: WRITE


## 6. Versionierung & Time Travel

Da jede Version im Transaktionslog erhalten bleibt, kann DuckDB mit `AT (VERSION => n)` oder `AT (TIMESTAMP => '...')` jeden historischen Zustand abfragen – **ohne Datenverlust**.

### Delta-Tabelle anbinden

`ATTACH ... (TYPE delta)` registriert die Delta-Tabelle in DuckDB, sodass sie wie jede normale Tabelle per SQL abfragbar ist. Erst dadurch funktioniert die Time-Travel-Syntax in den nächsten Zellen.

In [36]:
con.sql("ATTACH 'data/flights_delta' AS flights_delta (TYPE delta)")

### Time Travel → Version 0

`AT (VERSION => 0)` springt zum allerersten Snapshot zurück: 360 Zeilen, nur Q1. Die heutigen Daten sind nicht weg – sie sind nur in einer späteren Version. Das ist das Kernversprechen von Versionierung: jeder historische Zustand bleibt rekonstruierbar.

In [37]:
# Version 0 – nur Q1
con.sql("""
SELECT COUNT(*) AS zeilen, MIN(datum) AS von, MAX(datum) AS bis, 'Version 0 – nur Q1' AS snapshot
FROM flights_delta AT (VERSION => 0)
""")

┌────────┬─────────────────────┬─────────────────────┬────────────────────┐
│ zeilen │         von         │         bis         │      snapshot      │
│ int64  │      timestamp      │      timestamp      │      varchar       │
├────────┼─────────────────────┼─────────────────────┼────────────────────┤
│    360 │ 2025-01-01 00:00:00 │ 2025-03-31 18:00:00 │ Version 0 – nur Q1 │
└────────┴─────────────────────┴─────────────────────┴────────────────────┘

### Time Travel → Version 1

Version 1: Q1 + Q2 zusammen, 724 Zeilen. Wir sehen exakt den Zustand der Tabelle nach dem zweiten Append – als ob Version 2 nie passiert wäre.

In [38]:
# Version 1 – Q1 + Q2
con.sql("""
SELECT COUNT(*) AS zeilen, MIN(datum) AS von, MAX(datum) AS bis, 'Version 1 – Q1+Q2' AS snapshot
FROM flights_delta AT (VERSION => 1)
""")

┌────────┬─────────────────────┬─────────────────────┬───────────────────┐
│ zeilen │         von         │         bis         │     snapshot      │
│ int64  │      timestamp      │      timestamp      │      varchar      │
├────────┼─────────────────────┼─────────────────────┼───────────────────┤
│    724 │ 2025-01-01 00:00:00 │ 2025-06-30 18:00:00 │ Version 1 – Q1+Q2 │
└────────┴─────────────────────┴─────────────────────┴───────────────────┘

### Time Travel → Version 2

Version 2 ist der aktuelle Stand: alle 1000 Zeilen aus Q1 bis Q3. Ohne `AT (VERSION => ...)` würde DuckDB standardmäßig diese neueste Version liefern.

In [39]:
# Version 2 (aktuell) – Q1 + Q2 + Q3
con.sql("""
SELECT COUNT(*) AS zeilen, MIN(datum) AS von, MAX(datum) AS bis, 'Version 2 – Q1+Q2+Q3' AS snapshot
FROM flights_delta AT (VERSION => 2)
""")

┌────────┬─────────────────────┬─────────────────────┬──────────────────────┐
│ zeilen │         von         │         bis         │       snapshot       │
│ int64  │      timestamp      │      timestamp      │       varchar        │
├────────┼─────────────────────┼─────────────────────┼──────────────────────┤
│   1000 │ 2025-01-01 00:00:00 │ 2025-09-07 18:00:00 │ Version 2 – Q1+Q2+Q3 │
└────────┴─────────────────────┴─────────────────────┴──────────────────────┘

## Fazit

| Anforderung | Demonstriert |
|---|---|
| DuckDB ohne Docker/JVM | `pip install duckdb`, kein Cluster nötig |
| CSV-Ingestion | `flights.csv` → DuckDB-Tabelle |
| JSON-Ingestion | `airports.json` mit UNNEST |
| Parquet-Konvertierung | Partitioniert (year/month) + Einzeldatei |
| Delta-Konvertierung | `write_deltalake()` mit drei Versionen |
| Analytics SQL | JOIN, GROUP BY, RANK(), Window-Funktion, % |
| ACID-Transaktionen | Transaktionslog + History-Ausgabe |
| Versionierung | v0 → v1 → v2 mit Zeilencount |
| Time Travel VERSION | `AT (VERSION => 0/1/2)` |
| Time Travel TIMESTAMP | `AT (TIMESTAMP => '...')` |
| Spark vs. DuckDB | Vergleichstabelle in der Einleitung |
| ML-Workloads | Erklärung offene Formate in Einleitung |

In der Produktion würde Apache Spark dieselben Delta-Tabellen auf S3/ADLS verwalten. DuckDB eignet sich für lokale Entwicklung, Analyse und leichte Pipelines – ohne Infrastruktur-Overhead.

### Blick in den Transaktionslog – erste Zeile

Wir öffnen die erste Datei in `_delta_log/` und lesen nur die erste Zeile. Das ist der `commitInfo`-Block – Delta speichert hier *Metadaten über die Transaktion*: Zeitstempel, Operation, schreibendes Tool und Metriken (1 Datei hinzugefügt, 360 Zeilen). Das ist der konkrete Inhalt, der ACID hinter den Kulissen ausmacht.

In [40]:
from pathlib import Path
import json

log_dir = Path("data/flights_delta/_delta_log")
first_log = sorted(log_dir.glob("*.json"))[0]

with first_log.open("r", encoding="utf-8") as f:
    first_line = f.readline().strip()

log_entry = json.loads(first_line)
display(log_entry)

{'commitInfo': {'timestamp': 1777886706892,
  'operation': 'WRITE',
  'operationParameters': {'mode': 'ErrorIfExists'},
  'engineInfo': 'delta-rs:py-1.5.1',
  'clientVersion': 'delta-rs.py-1.5.1',
  'operationMetrics': {'execution_time_ms': 2,
   'num_added_files': 1,
   'num_added_rows': 360,
   'num_partitions': 0,
   'num_removed_files': 0}}}

### Kompletter Log-Eintrag

Dieselbe Log-Datei, aber alle Zeilen. Eine Delta-Commit-Datei besteht aus mehreren JSON-Objekten: `commitInfo` (was passiert ist), `protocol` (Format-Version), `metaData` (Schema und Spalten-Definition) und `add` (welche Parquet-Datei mit welchen Min/Max-Statistiken hinzugefügt wurde). Die Statistiken in `add.stats` ermöglichen es Delta später, irrelevante Dateien beim Lesen zu überspringen – **Data Skipping**.

In [41]:
from pathlib import Path
import json

log_dir = Path("data/flights_delta/_delta_log")
first_log = sorted(log_dir.glob("*.json"))[0]

with first_log.open("r", encoding="utf-8") as f:
    log_entries = [json.loads(line) for line in f if line.strip()]

display(log_entries)

[{'commitInfo': {'timestamp': 1777886706892,
   'operation': 'WRITE',
   'operationParameters': {'mode': 'ErrorIfExists'},
   'engineInfo': 'delta-rs:py-1.5.1',
   'clientVersion': 'delta-rs.py-1.5.1',
   'operationMetrics': {'execution_time_ms': 2,
    'num_added_files': 1,
    'num_added_rows': 360,
    'num_partitions': 0,
    'num_removed_files': 0}}},
 {'protocol': {'minReaderVersion': 3,
   'minWriterVersion': 7,
   'readerFeatures': ['timestampNtz'],
   'writerFeatures': ['timestampNtz']}},
 {'metaData': {'id': 'f6674968-e02e-4232-9854-8abf3d053bcf',
   'name': None,
   'description': None,
   'format': {'provider': 'parquet', 'options': {}},
   'schemaString': '{"type":"struct","fields":[{"name":"flight_id","type":"string","nullable":true,"metadata":{}},{"name":"source","type":"string","nullable":true,"metadata":{}},{"name":"dest","type":"string","nullable":true,"metadata":{}},{"name":"passenger","type":"string","nullable":true,"metadata":{}},{"name":"datum","type":"timestamp_n